# 🚀 Optimize Edilmiş Deprem Bina/Yol Hasar Tespiti — Model Eğitimi (v2)

Bu notebook, önceki eğitimdeki **veri sızıntısı, negatif örnek yokluğu ve focal α ayarı** sorunlarını
düzelterek daha dürüst ve daha iyi genelleyen bir model eğitir.

### Bu sürümdeki iyileştirmeler (aktarım belgesi A-16 / A-18 / A-19):
- ✅ **Sızıntı azaltılmış bölme** — rastgele `train_test_split` KALDIRILDI:
  - KATE-CD: **resmî train/validation/test bölmeleri** (404/44/38) kullanılıyor (eskiden yalnız `train` alınıp rastgele bölünüyordu)
  - MEDBFS: **sahne-bazlı bölme** (GeoTIFF sahne-orijinine göre gruplanıp bölme) — aynı sahnenin komşu karoları artık aynı bölmede
- ✅ **Negatif (hasarsız) örnekler eklendi** — KATE-CD ve MEDBFS'in **deprem öncesi (pre-disaster)** görüntüleri tüm-sıfır maskeyle negatif olarak katılıyor (eskiden 842/842 karo hasarlıydı, model hasarsız sahne hiç görmüyordu → yanlış alarm ölçülemiyordu)
- ✅ **Focal α taraması** — `α ∈ {0,25 · 0,5 · 0,75}`; α=0,25 hasar piksellerini arka plandan **3× az** ağırlıklandırıyordu (A-19), doğru α doğrulama kümesinde seçiliyor
- ✅ **Global piksel IoU** — doğrulama IoU'su artık tüm pikseller üzerinden birikimli hesaplanıyor (makaledeki "hasar IoU'su tüm sınama pikselleri üzerinden" tanımıyla uyumlu; batch-ortalaması değil)
- ✅ **`ignore_index=255` altyapısı** — maske değeri 255 kayıptan ve ölçütten hariç tutuluyor (QQB'deki yanlış etiketli binalar / bulut-gölge için hazır; bu iki veri setinde şu an 255 yok, altyapı gelecek veri için)
- ✅ ImageNet normalizasyonu · Focal+Dice birleşik loss · CosineAnnealing · uydu-özel augmentation · mit_b4 encoder (korundu)

> ### ⚠️ Mekânsal bölme hakkında dürüst kısıt
> Makalenin hedeflediği **2 km × 2 km UTM blok** bölmesi bu notebook'ta **birebir yapılamıyor**: HuggingFace
> `CSCRS/kate-cd` sürümünde karoların **konum/koordinat bilgisi yok** (aktarım belgesi A-16). Gerçek 2 km-blok
> bölmesi KATE-PD'nin CodeOcean ızgara dosyalarını (`grid_maxar.geojson` vb.) ve `QuakeMind_veri/araclar/bolme.py`
> paketini gerektirir — bunlar bu depoda değil. Burada uygulanan bölme **sızıntıyı azaltan bir yaklaşım**tır
> (KATE-CD resmî bölmesi + MEDBFS sahne-bazlı bölme), tam sızıntısız 2 km-blok bölmesi değildir. KATE-CD resmî
> bölmeleri de karo düzeyindedir (mekânsal değil); tam protokol için `QuakeMind_veri/` paketiyle çalışılmalıdır.

**ÖNEMLİ:** Üst menüden `Çalışma Zamanı` → `Çalışma zamanı türünü değiştir` → **T4 GPU** seçin.

## 1. Kütüphaneler

In [ ]:
!pip install -q datasets opencv-python numpy tqdm scikit-learn torch torchvision segmentation-models-pytorch albumentations tifffile

## 2. Dizinleri Oluştur

In [ ]:
import os
import re
import numpy as np
import cv2

# Karolar hem KATE-CD hem MEDBFS'te zaten 512x512 → ayrı "tiling" adımı gerekmez.
IMG_DIR  = 'data/images/'
MASK_DIR = 'data/masks/'
MODEL_DIR = 'models/'
for d in [IMG_DIR, MASK_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# ────────────────────────────────────────
# Maske düzeni:  0 = arka plan · 1 = hasar · 255 = YOK SAY (ignore_index)
# ────────────────────────────────────────
IGNORE_INDEX = 255

# ────────────────────────────────────────
# Dosya adı şeması — bölme/kutupluluk/sahne bilgisini adın içine gömüyoruz.
# Böylece hücreler yeniden çalıştırılsa bile manifest yeniden türetilebilir (idempotent).
#   {source}__{scene}__{split}__{polarity}__{ident}.png
#   source   : kate | medbfs
#   scene    : none (KATE-CD)  |  part3 / s0 vb. (MEDBFS sahne anahtarı)
#   split    : train | validation | test
#   polarity : pos (hasar) | neg (öncesi/hasarsız)
# ────────────────────────────────────────
def make_name(source, scene, split, polarity, ident):
    scene = str(scene).replace('__', '_')
    ident = str(ident).replace('__', '_')
    return f'{source}__{scene}__{split}__{polarity}__{ident}.png'

def parse_name(fn):
    stem = fn[:-4] if fn.lower().endswith('.png') else fn
    p = stem.split('__')
    if len(p) < 5:
        return None
    return {'source': p[0], 'scene': p[1], 'split': p[2],
            'polarity': p[3], 'ident': '__'.join(p[4:]), 'file': fn}

print('✅ Klasörler hazır.  Maske düzeni: 0=arka plan, 1=hasar, 255=yok say')

## 3. KATE-CD Veri Seti (resmî bölme + öncesi görüntü = negatif)
- **Resmî `train`/`validation`/`test` bölmeleri** kullanılıyor (404/44/38) — rastgele yeniden bölme yok.
- Her karonun **deprem sonrası** görüntüsü pozitif; **deprem öncesi (`pre_image`)** görüntüsü tüm-sıfır maskeyle **negatif** olarak *aynı bölmeye* ekleniyor (aktarım belgesi A-16: KATE-CD'nin asıl katkısı öncesi görüntülerdir; sonrası görüntüleri KATE-PD ile aynıdır).

In [ ]:
from datasets import load_dataset

# Negatif (öncesi) örnekleri ekle. 1.0 = her sonrası karo için bir öncesi negatif (≈%50 negatif).
# Daha az negatif istenirse 0.0–1.0 arası küçült.
INCLUDE_KATE_NEG = True
KATE_NEG_KEEP = 1.0

print('KATE-CD çekiliyor (resmî train/validation/test bölmeleri)...')
try:
    ds_kate = load_dataset('CSCRS/kate-cd')
except Exception as e:
    print(f'Hata: {e}. Alternatif deneniyor...')
    ds_kate = load_dataset('hkayabilisim/kate-cd')

def _to_bgr(pil):
    arr = np.array(pil)
    if arr.ndim == 3 and arr.shape[2] == 3:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    return arr

def _save(img_bgr, mask_u8, name):
    cv2.imwrite(os.path.join(IMG_DIR, name), img_bgr)
    cv2.imwrite(os.path.join(MASK_DIR, name), mask_u8)

kate_pos = kate_neg = 0
rng = np.random.default_rng(42)

for split in ['train', 'validation', 'test']:
    if split not in ds_kate:
        continue
    for i, v in enumerate(ds_kate[split]):
        post = v.get('post_image') or v.get('image')
        pre  = v.get('pre_image')
        lab  = v.get('label') or v.get('mask')
        if post is None or lab is None:
            continue

        m = np.array(lab)
        if m.ndim > 2:
            m = cv2.cvtColor(m, cv2.COLOR_RGB2GRAY)
        m = np.where(m > 0, 1, 0).astype(np.uint8)          # 0/1 (255 = ignore ayrılmıştır)

        post_bgr = _to_bgr(post)
        if post_bgr.shape[:2] == m.shape[:2]:
            _save(post_bgr, m, make_name('kate', 'none', split, 'pos', i))
            kate_pos += 1

        # Öncesi görüntü → tüm-sıfır maskeli negatif, AYNI bölmede (sızıntı yok)
        if INCLUDE_KATE_NEG and pre is not None and rng.random() < KATE_NEG_KEEP:
            pre_bgr = _to_bgr(pre)
            zero = np.zeros(pre_bgr.shape[:2], np.uint8)
            _save(pre_bgr, zero, make_name('kate', 'none', split, 'neg', i))
            kate_neg += 1

print(f'✅ KATE-CD: {kate_pos} pozitif (sonrası) + {kate_neg} negatif (öncesi) kaydedildi.')

## 4. MEDBFS Veri Seti (sahne-bazlı bölme + öncesi görüntü = negatif)
- MEDBFS karoları **6 büyük sahneden** yarı-örtüşmeli kesilmiş (aktarım belgesi A-16). Karo düzeyinde rastgele
  bölme, aynı sahnenin komşu karolarını eğitim ve sınamaya dağıtarak **sızıntı** yaratıyordu (1953 örtüşen çift).
- Burada her karonun **sahne-orijini** GeoTIFF `ModelTiepoint` etiketinden okunup karolar sahnelere gruplanıyor;
  bölme **sahne bütünlüğü korunarak** (~%80/10/10) yapılıyor → aynı sahne yalnız tek bölmede.
- **Deprem öncesi (Pre-disaster)** görüntüler tüm-sıfır maskeyle negatif olarak *aynı bölmeye* ekleniyor.
- Sahne-orijini okunamazsa (georeference yoksa) güvenli geri-dönüş: tüm MEDBFS **eğitime** konur (val/test'e sızmaz).

In [ ]:
import urllib.request
import zipfile
from tqdm import tqdm
import tifffile

INCLUDE_MEDBFS_NEG = True     # Pre-disaster görüntüleri negatif olarak ekle
MEDBFS_NEG_KEEP = 1.0

dataset_url = 'https://zenodo.org/records/11647956/files/Turkey.zip?download=1'
zip_path = '/content/Turkey.zip'
extract_path = '/content/MEDBFS_Turkey'

if not os.path.exists(zip_path):
    print('MedBFS İndiriliyor... (~3.3GB)')
    urllib.request.urlretrieve(dataset_url, zip_path)
    print('İndirme tamamlandı.')
if not os.path.exists(extract_path):
    print('MedBFS Çıkarılıyor...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print('Çıkarma tamamlandı.')

medbfs_post_dir = '/content/MEDBFS_Turkey/Turkey/Image/Post-disaster'
medbfs_pre_dir  = '/content/MEDBFS_Turkey/Turkey/Image/Pre-disaster'
medbfs_lbl_dir  = '/content/MEDBFS_Turkey/Turkey/Label'

# ── 1) Sahne-orijinini GeoTIFF ModelTiepoint etiketinden oku ──
def scene_origin(path):
    try:
        with tifffile.TiffFile(path) as t:
            tags = t.pages[0].tags
            tp = None
            for key in ('ModelTiepointTag', 33922):
                try:
                    tp = tags.get(key)
                except Exception:
                    tp = None
                if tp is not None:
                    break
            if tp is None:
                return None
            v = list(tp.value)
            if len(v) >= 6:
                return (round(float(v[3]), 1), round(float(v[4]), 1))  # sahne (x0, y0)
    except Exception:
        return None
    return None

post_files = sorted(f for f in os.listdir(medbfs_post_dir)
                    if f.endswith('.tif') and os.path.exists(os.path.join(medbfs_lbl_dir, f)))
print(f'MEDBFS sonrası karo (etiketli): {len(post_files)}')

origins = {f: scene_origin(os.path.join(medbfs_post_dir, f)) for f in tqdm(post_files, desc='Sahne-orijini okunuyor')}
distinct = sorted({o for o in origins.values() if o is not None})
n_scene = len(distinct)

# ── 2) Karoyu sahneye ata ──
USE_SCENE_SPLIT = 2 <= n_scene <= 40
if USE_SCENE_SPLIT:
    origin_to_scene = {o: f's{i}' for i, o in enumerate(distinct)}
    file_scene = {f: origin_to_scene.get(origins[f], 'sX') for f in post_files}
    print(f'✅ {n_scene} sahne saptandı (GeoTIFF sahne-orijini). Sahne-bazlı bölme kullanılacak.')
else:
    file_scene = {f: 's0' for f in post_files}
    print(f'⚠️ Sahne-orijini güvenilir gruplama vermedi (distinct={n_scene}). '
          f'GERİ-DÖNÜŞ: tüm MEDBFS EĞİTİME konacak (val/test yalnız KATE-CD\'den).')

# ── 3) Sahne → bölme (aç-gözlü, %80/10/10 hedefine göre; sahne bütünlüğü korunur) ──
from collections import Counter, defaultdict
scene_counts = Counter(file_scene.values())
scene_split = {}
if USE_SCENE_SPLIT:
    targets = {'train': 0.80, 'validation': 0.10, 'test': 0.10}
    assigned = {'train': 0, 'validation': 0, 'test': 0}
    total_tiles = sum(scene_counts.values())
    for scene, cnt in scene_counts.most_common():            # büyük sahneden küçüğe
        # hedefine göre en açıkta kalan bölmeye ver
        deficit = {s: targets[s] - (assigned[s] / max(total_tiles, 1)) for s in targets}
        pick = max(deficit, key=deficit.get)
        scene_split[scene] = pick
        assigned[pick] += cnt
else:
    scene_split = {s: 'train' for s in scene_counts}

print('Sahne dağılımı (sahne: karo → bölme):')
for scene, cnt in scene_counts.most_common():
    print(f'   {scene}: {cnt} karo → {scene_split[scene]}')

# ── 4) Kaydet: pozitif (sonrası+etiket) + negatif (öncesi+sıfır) ──
def _save(img_bgr, mask_u8, name):
    cv2.imwrite(os.path.join(IMG_DIR, name), img_bgr)
    cv2.imwrite(os.path.join(MASK_DIR, name), mask_u8)

rng_m = np.random.default_rng(43)
medbfs_pos = medbfs_neg = 0
for f in tqdm(post_files, desc='MEDBFS kaydediliyor'):
    scene = file_scene[f]
    split = scene_split[scene]
    img = cv2.imread(os.path.join(medbfs_post_dir, f))
    mask = cv2.imread(os.path.join(medbfs_lbl_dir, f), cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None or img.shape[:2] != mask.shape[:2]:
        continue
    mask = np.where(mask > 0, 1, 0).astype(np.uint8)
    base = os.path.splitext(f)[0]
    _save(img, mask, make_name('medbfs', scene, split, 'pos', base))
    medbfs_pos += 1

    pre_path = os.path.join(medbfs_pre_dir, f)
    if INCLUDE_MEDBFS_NEG and os.path.exists(pre_path) and rng_m.random() < MEDBFS_NEG_KEEP:
        pre = cv2.imread(pre_path)
        if pre is not None:
            zero = np.zeros(pre.shape[:2], np.uint8)
            _save(pre, zero, make_name('medbfs', scene, split, 'neg', base))
            medbfs_neg += 1

print(f'✅ MEDBFS: {medbfs_pos} pozitif (sonrası) + {medbfs_neg} negatif (öncesi) eklendi.')

## 5. Manifest ve Bölme Raporu
KATE-CD ve MEDBFS karoları **zaten 512×512** olduğundan ayrı parçalama (tiling) adımı yok. Bu hücre dosya
adlarından manifest'i türetir, **bölme × kutupluluk × kaynak** dağılımını yazdırır ve *hiçbir MEDBFS sahnesinin
bölmeler arasına dağılmadığını* (sızıntı yok) doğrular.

In [ ]:
import csv
from collections import Counter, defaultdict

# ── Manifest'i dosya adlarından türet (idempotent) ──
manifest = []
for fn in sorted(os.listdir(IMG_DIR)):
    rec = parse_name(fn)
    if rec is None:
        continue
    if not os.path.exists(os.path.join(MASK_DIR, fn)):
        continue
    manifest.append(rec)

with open('data/manifest.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=['file', 'source', 'scene', 'split', 'polarity', 'ident'])
    w.writeheader()
    for r in manifest:
        w.writerow({k: r[k] for k in w.fieldnames})

total = len(manifest)
print(f'📦 Manifest: {total} karo  →  data/manifest.csv\n')

# ── Bölme × kutupluluk dağılımı ──
by_split = defaultdict(lambda: Counter())
for r in manifest:
    by_split[r['split']][r['polarity']] += 1
print(f'{"bölme":<12}{"pozitif":>9}{"negatif":>9}{"toplam":>9}{"neg %":>8}')
for split in ['train', 'validation', 'test']:
    c = by_split[split]
    pos, neg = c['pos'], c['neg']
    tot = pos + neg
    frac = (100 * neg / tot) if tot else 0
    print(f'{split:<12}{pos:>9}{neg:>9}{tot:>9}{frac:>7.1f}%')

# ── Kaynak dağılımı ──
by_source = Counter(r['source'] for r in manifest)
print('\nKaynak:', dict(by_source))

# ── Sızıntı doğrulaması: bir MEDBFS sahnesi birden çok bölmede olmamalı ──
scene_splits = defaultdict(set)
for r in manifest:
    if r['source'] == 'medbfs':
        scene_splits[r['scene']].add(r['split'])
leaks = {sc: sorted(sp) for sc, sp in scene_splits.items() if len(sp) > 1}
if leaks:
    print('\n❌ SIZINTI: aşağıdaki sahneler birden çok bölmede:', leaks)
else:
    print('\n✅ Sızıntı yok: her MEDBFS sahnesi tek bölmede.')

if total == 0:
    print('\n⚠️ HATA: veri bulunamadı — 3. ve 4. hücreleri çalıştırdınız mı?')

## 6. Model Eğitimi — Focal α taraması
- **Encoder:** mit_b4 (ImageNet pretrained) · **Loss:** Focal + Dice (0,5 + 0,5, `ignore_index=255`)
- **Optimizer:** AdamW + CosineAnnealingWarmRestarts · **Norm.:** ImageNet
- **Bölme:** manifest'ten (rastgele bölme YOK) · **Metrik:** global piksel IoU (255 hariç)
- **Focal α taraması:** `α ∈ {0,25 · 0,5 · 0,75}` (A-19) — en iyi α doğrulama IoU'suna göre seçilir.

> ⏱️ **Süre uyarısı:** α taraması modeli **her α için baştan eğitir** (≈3× süre, T4'te ~3–4 saat).
> Tek bir değerle koşmak için `FOCAL_ALPHAS = [0.75]` yapın. Varsayılan değeri kalıcı olarak değiştirmeden
> önce belge (A-19) çoklu tohumla doğrulama öneriyor.

In [ ]:
import shutil
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torch.optim as optim

# ══════════════════════════════════════════════════════════
#  Ayarlar
# ══════════════════════════════════════════════════════════
FOCAL_ALPHAS = [0.25, 0.5, 0.75]   # A-19 taraması. Tek değer → tarama yapılmaz.
GAMMA   = 2.0
EPOCHS  = 40
PATIENCE = 8
BATCH   = 8
LR      = 2e-4
SEED    = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Dataset — maske {0,1,255} korunur (255 = ignore); threshold YOK ──
class DepremDataset(Dataset):
    def __init__(self, files, img_dir, mask_dir, transform=None):
        self.files = files; self.img_dir = img_dir; self.mask_dir = mask_dir; self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        name = self.files[idx]
        image = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, name)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(os.path.join(self.mask_dir, name), cv2.IMREAD_GRAYSCALE).astype(np.float32)
        if self.transform:
            aug = self.transform(image=image, mask=mask)   # mask için INTER_NEAREST → {0,1,255} korunur
            image, mask = aug['image'], aug['mask']
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            for c in range(3):
                image[c] = (image[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
            mask = torch.from_numpy(mask)
        return image, mask.long()

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5,
                       border_mode=cv2.BORDER_REFLECT_101),
    A.RandomResizedCrop(size=(512, 512), scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.3),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.15),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
])
val_transform = A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

# ── Bölmeyi manifest'ten al (rastgele bölme YOK) ──
def files_for(split):
    return [r['file'] for r in manifest if r['split'] == split]

train_files, val_files, test_files = files_for('train'), files_for('validation'), files_for('test')
print(f'Bölme  → train {len(train_files)} · val {len(val_files)} · test {len(test_files)}')
if not val_files:
    print('⚠️ Doğrulama kümesi boş — MEDBFS sahne-orijini okunamamış olabilir; KATE-CD resmî val/test kullanılır.')

train_loader = DataLoader(DepremDataset(train_files, IMG_DIR, MASK_DIR, train_transform),
                          batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(DepremDataset(val_files, IMG_DIR, MASK_DIR, val_transform),
                          batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device}')

# ── Global piksel IoU (255 hariç) ──
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    inter = union = 0.0; tot_loss = 0.0; nb = 0
    for images, masks in loader:
        images = images.to(device)
        m = masks.to(device).unsqueeze(1).float()
        out = model(images)
        tot_loss += loss_fn(out, m).item(); nb += 1
        valid = (m != IGNORE_INDEX)
        tgt   = (m == 1) & valid
        pred  = (torch.sigmoid(out) > 0.5) & valid
        inter += (pred & tgt).sum().item()
        union += (pred | tgt).sum().item()
    iou = inter / union if union > 0 else 1.0
    return tot_loss / max(nb, 1), iou

# ══════════════════════════════════════════════════════════
#  Focal α taraması
# ══════════════════════════════════════════════════════════
sweep_results = []
best_overall = {'alpha': None, 'iou': -1.0, 'path': None}

for alpha in FOCAL_ALPHAS:
    print('\n' + '=' * 70)
    print(f'🔬 FOCAL α = {alpha}  (γ={GAMMA})  — model baştan eğitiliyor')
    print('=' * 70)

    model = smp.Segformer(encoder_name='mit_b4', encoder_weights='imagenet',
                          in_channels=3, classes=1).to(device)
    dice_fn  = smp.losses.DiceLoss(mode='binary', ignore_index=IGNORE_INDEX)
    focal_fn = smp.losses.FocalLoss(mode='binary', alpha=alpha, gamma=GAMMA, ignore_index=IGNORE_INDEX)
    def loss_fn(pred, target, _d=dice_fn, _f=focal_fn):
        return 0.5 * _d(pred, target) + 0.5 * _f(pred, target)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

    save_path_a = f'{MODEL_DIR}optimized_mitb4_focal_dice_a{alpha:g}.pth'
    best_iou = 0.0; best_loss = float('inf'); no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        run_loss = 0.0; tr_inter = tr_union = 0.0
        pbar = tqdm(train_loader, desc=f'α{alpha} E{epoch+1}/{EPOCHS}')
        for images, masks in pbar:
            images = images.to(device)
            m = masks.to(device).unsqueeze(1).float()
            optimizer.zero_grad()
            out = model(images)
            loss = loss_fn(out, m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            run_loss += loss.item()
            with torch.no_grad():
                valid = (m != IGNORE_INDEX); tgt = (m == 1) & valid
                pred = (torch.sigmoid(out) > 0.5) & valid
                tr_inter += (pred & tgt).sum().item(); tr_union += (pred | tgt).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        scheduler.step()

        tr_iou = tr_inter / tr_union if tr_union > 0 else 1.0
        val_loss, val_iou = evaluate(model, val_loader, loss_fn)
        lr_now = optimizer.param_groups[0]['lr']
        print(f'👉 α{alpha} E{epoch+1} | Train Loss {run_loss/len(train_loader):.4f} | '
              f'Val Loss {val_loss:.4f} | Train IoU {tr_iou:.4f} | Val IoU {val_iou:.4f} | LR {lr_now:.6f}')

        if val_iou > best_iou:
            best_iou, best_loss, no_improve = val_iou, val_loss, 0
            torch.save(model.state_dict(), save_path_a)
            print(f'   ✨ Yeni en iyi (α={alpha}) Val IoU {best_iou:.4f} → {save_path_a}')
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'   ⏳ Early stopping (α={alpha}). En iyi Val IoU {best_iou:.4f}')
                break

    sweep_results.append({'alpha': alpha, 'val_iou': best_iou, 'val_loss': best_loss, 'path': save_path_a})
    if best_iou > best_overall['iou']:
        best_overall = {'alpha': alpha, 'iou': best_iou, 'path': save_path_a}

    del model, optimizer, scheduler
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# ── Özet + en iyi modeli kararlı ada kopyala ──
print('\n' + '=' * 70)
print('🏁 FOCAL α TARAMASI ÖZETİ')
print(f'{"alpha":>8}{"val IoU":>12}{"val loss":>12}')
for r in sorted(sweep_results, key=lambda x: -x['val_iou']):
    print(f'{r["alpha"]:>8}{r["val_iou"]:>12.4f}{r["val_loss"]:>12.4f}')

STABLE_PATH = f'{MODEL_DIR}optimized_mitb4_focal_dice.pth'
if best_overall['path']:
    shutil.copy(best_overall['path'], STABLE_PATH)
    print(f'\n🏆 En iyi: α={best_overall["alpha"]} · Val IoU {best_overall["iou"]:.4f}')
    print(f'   Kararlı ad: {STABLE_PATH}')

## 7. Sınama Değerlendirmesi + Modeli İndir
En iyi (α seçilmiş) modeli **sınama kümesinde** global piksel IoU + kesinlik/duyarlılık/F1 ile ölçer, sonra
ağırlıkları indirir.

**NOT:** İndirilen `.pth` dosyasını `deprem_yol_projesi/models/` klasörüne koyup `app.py` sidebar'ından yolu güncelleyin.

In [ ]:
from google.colab import files

# ── Sınama kümesinde en iyi modeli değerlendir (global; 255 hariç) ──
if test_files:
    test_loader = DataLoader(DepremDataset(test_files, IMG_DIR, MASK_DIR, val_transform),
                             batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
    best_model = smp.Segformer(encoder_name='mit_b4', encoder_weights=None,
                               in_channels=3, classes=1).to(device)
    best_model.load_state_dict(torch.load(STABLE_PATH, map_location=device))
    best_model.eval()

    tp = fp = fn = 0.0
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc='Sınama'):
            images = images.to(device)
            m = masks.to(device).unsqueeze(1).float()
            valid = (m != IGNORE_INDEX)
            tgt   = (m == 1) & valid
            pred  = (torch.sigmoid(best_model(images)) > 0.5) & valid
            tp += (pred & tgt).sum().item()
            fp += (pred & ~tgt & valid).sum().item()
            fn += (~pred & tgt).sum().item()

    iou  = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float('nan')
    prec = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    rec  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')
    print(f'\n📊 SINAMA (α={best_overall["alpha"]}): '
          f'IoU_hasar {iou:.4f} · Kesinlik {prec:.4f} · Duyarlılık {rec:.4f} · F1 {f1:.4f}')
else:
    print('⚠️ Sınama kümesi boş — test değerlendirmesi atlandı.')

# ── En iyi modeli indir ──
if os.path.exists(STABLE_PATH):
    print(f'\nModel boyutu: {os.path.getsize(STABLE_PATH) / 1e6:.1f} MB')
    files.download(STABLE_PATH)
else:
    print('Model dosyası bulunamadı!')